# 3. Two-Agent Workflow: Investigate + Review

### Scenario (continued)
> *Customer upgraded to MongoDB 9.0. Reports: slow aggregations, memory errors on writes, change-stream lag.*

### What this notebook shows
Two LLM calls with **different roles**:
- **Investigator Agent** — runs diagnostic tools, gathers evidence
- **TS Reviewer Agent** — validates findings against release notes, produces a structured report

The handoff is **our Python code** passing notes from one to the other. This is role separation, not autonomous swarm.

Run cells top to bottom.

## Setup — imports and settings

In [1]:
import ollama, requests, re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
# TF-IDF for document search (same as notebooks 1 and 2).

MODEL = "qwen2.5:3b"
OUTPUT_FILE = "support_findings.md"

DOC_PATHS = ["9.0_notes.md", "9.0_compat.md"]
# Two clean markdown sources.

print(f"Model: {MODEL}  |  Output: {OUTPUT_FILE}")

Model: qwen2.5:3b  |  Output: support_findings.md


## Load docs and define diagnostic functions

Same building blocks as notebook 2.

In [2]:
# ---- Load docs ----
doc_texts = {}
for path in DOC_PATHS:
    with open(path, encoding="utf-8") as f:
        doc_texts[path] = f.read()
    print(f"Loaded: {path} ({len(doc_texts[path])} chars)")


# ---- Document search ----
def search_docs(query, max_chunks=3):
    all_chunks = []
    for name, text in doc_texts.items():
        for i in range(0, len(text), 600):
            chunk = text[i:i + 800]
            if len(chunk.strip()) > 50:
                all_chunks.append((name, chunk.strip()))
    vec = TfidfVectorizer(stop_words="english")
    m = vec.fit_transform([c[1] for c in all_chunks] + [query])
    scores = cosine_similarity(m[-1], m[:-1])[0]
    top = scores.argsort()[::-1][:max_chunks]
    results = []
    for idx in top:
        doc_name, chunk_text = all_chunks[idx]
        results.append(f"[From: {doc_name}]\n{chunk_text[:500]}")
    return "\n\n---\n\n".join(results)


# ---- Diagnostic tools (same samples as notebook 2) ----
def parse_explain():
    wp = {"queryPlan": {"stage": "FETCH", "inputStage": {"stage": "IXSCAN",
        "indexName": "customerId_1_createdAt_1"}},
        "slotBasedPlan": {"stages": "group [s12 = sum(s8)]"}}
    es = {"nReturned": 42, "executionTimeMillis": 1847,
          "totalKeysExamined": 1284000, "totalDocsExamined": 1284000,
          "peakTrackedMemBytes": 118734848}
    engine = "SBE" if "slotBasedPlan" in wp else "Classic"
    ratio = es["totalDocsExamined"] / max(es["nReturned"], 1)
    mem_mb = es["peakTrackedMemBytes"] / (1024 * 1024)
    return (f"Engine: {engine} | Root: {wp['queryPlan']['stage']} | "
            f"Examined/Returned: {ratio:.0f}x | Time: {es['executionTimeMillis']}ms | "
            f"Peak mem: {mem_mb:.0f}MB | Version: 9.0.0")


def check_queryStats():
    return (
        "Operation: update | Examined: 984,000 | Updated: 18,420 | "
        "Scan ratio: 53x | Delinquent acquisitions: 29/37 | "
        "Errors: code 146 (8 times), code 292 (4 times)"
    )


def read_serverStatus():
    return (
        "Host: rs0-primary-0 | MongoDB 9.0.0 | "
        "Memory limit: 1.0GB | Ops failed (memory): 137 | "
        "Spilled to disk: 4,812 | $lookup+$unwind SBE calls: 8,420 | "
        "Write conflicts: 53 | Total ops: 54,700"
    )


def classify_error(code):
    codes = {
        146: ("ExceededMemoryLimit", "Query hit the 9.0 per-operation memory guardrail (default 1GB or 20% RAM)."),
        292: ("QueryExceededMemoryLimitNoDiskUseAllowed", "Memory-intensive op can't spill because allowDiskUse is false."),
        509: ("TooManyOpenTransactions", "Exceeded max concurrent multi-doc transactions (default 10,000)."),
        485: ("InterruptedDueToTimeseriesUpgradeDowngrade", "Legacy time-series bucket access blocked after 9.0 upgrade."),
        491: ("CommandNotSupportedOnLegacyTimeseriesBucketsNamespace", "Use logical collection name, not system.buckets."),
    }
    info = codes.get(int(code))
    if info:
        return f"Error {code} — {info[0]}: {info[1]}"
    return f"Error {code} — Unknown code."


def get_weather(city="Dublin"):
    coords = {"Dublin": (53.3498, -6.2603), "London": (51.5074, -0.1278),
               "New York": (40.7128, -74.0060)}
    lat, lon = coords.get(city, coords["Dublin"])
    try:
        url = "https://api.open-meteo.com/v1/forecast"
        params = {"latitude": lat, "longitude": lon, "current": "temperature_2m,precipitation,wind_speed_10m"}
        d = requests.get(url, params=params, timeout=10).json()["current"]
        return f"{city}: {d['temperature_2m']}C, precipitation {d['precipitation']}mm, wind {d['wind_speed_10m']}km/h"
    except Exception as e:
        return f"Weather failed: {e}"


print("All diagnostic functions ready.")

Loaded: 9.0_notes.md (7649 chars)
Loaded: 9.0_compat.md (3782 chars)
All diagnostic functions ready.


## Agent 1 — The Investigator

Gathers raw evidence: searches docs, runs diagnostics, classifies errors. Outputs a structured evidence log.

In [3]:
print("=== INVESTIGATOR AGENT STARTED ===\n")
# The investigator runs all diagnostic tools, like a first responder collecting evidence.

doc_findings = search_docs("per-operation memory limit aggregation SBE change stream 9.0")
explain_result = parse_explain()
stats_result = check_queryStats()
server_result = read_serverStatus()
error_146 = classify_error(146)
error_292 = classify_error(292)
weather_result = get_weather("Dublin")

print("Evidence gathered:\n")
print(f"[Docs] {doc_findings[:200]}...")
print(f"[Explain] {explain_result}")
print(f"[QueryStats] {stats_result}")
print(f"[ServerStatus] {server_result}")
print(f"[Error 146] {error_146}")
print(f"[Error 292] {error_292}")
print(f"[Weather] {weather_result}")

# Investigator analyzes the evidence
inv_system = (
    "You are a MongoDB support investigator. Analyze the evidence below and produce "
    "structured findings with these sections:\n"
    "## Symptom\n## Likely 9.0 Changes\n## Evidence Summary\n## Risk Level (Low/Medium/High)\n"
    "## Preliminary Recommendation\n\n"
    "Be specific: mention error codes, engine types (SBE/Classic), memory limits, "
    "and version numbers. Keep each section 2-3 sentences."
)

inv_prompt = (
    f"DOCUMENTATION EVIDENCE:\n{doc_findings}\n\n"
    f"DIAGNOSTIC EVIDENCE:\n"
    f"- explain(): {explain_result}\n"
    f"- $queryStats: {stats_result}\n"
    f"- serverStatus: {server_result}\n"
    f"- Error 146: {error_146}\n"
    f"- Error 292: {error_292}\n\n"
    f"ADDITIONAL:\n- Weather: {weather_result}\n\n"
    "Produce the structured findings now."
)

inv_response = ollama.chat(model=MODEL, messages=[
    {"role": "system", "content": inv_system},
    {"role": "user", "content": inv_prompt},
])
# Single LLM call; the investigator analyzes everything at once.

raw_findings = inv_response["message"]["content"].strip()
print("\n=== INVESTIGATOR FINDINGS ===\n")
print(raw_findings)
# These raw findings will be handed off to the TS Reviewer.

=== INVESTIGATOR AGENT STARTED ===

Evidence gathered:

[Docs] [From: 9.0_notes.md]
of memory that a single query operation can use. By default, the limit is 1 gigabyte or 20% of the memory available to the server process, whichever is greater. Operations that ex...
[Explain] Engine: SBE | Root: FETCH | Examined/Returned: 30571x | Time: 1847ms | Peak mem: 113MB | Version: 9.0.0
[QueryStats] Operation: update | Examined: 984,000 | Updated: 18,420 | Scan ratio: 53x | Delinquent acquisitions: 29/37 | Errors: code 146 (8 times), code 292 (4 times)
[ServerStatus] Host: rs0-primary-0 | MongoDB 9.0.0 | Memory limit: 1.0GB | Ops failed (memory): 137 | Spilled to disk: 4,812 | $lookup+$unwind SBE calls: 8,420 | Write conflicts: 53 | Total ops: 54,700
[Error 146] Error 146 — ExceededMemoryLimit: Query hit the 9.0 per-operation memory guardrail (default 1GB or 20% RAM).
[Error 292] Error 292 — QueryExceededMemoryLimitNoDiskUseAllowed: Memory-intensive op can't spill because allowDiskUse is false.

## Handoff

The TS Reviewer receives **only** the investigator's raw findings — not the raw diagnostic data. This is the handoff between two specialized roles.

## Agent 2 — The TS Reviewer

Validates the investigator's findings, cross-references with release notes, rejects unsafe recommendations, and produces a final structured report.

In [4]:
print("=== TS REVIEWER AGENT STARTED ===\n")
# The reviewer is the second pair of eyes, like a senior engineer validating findings.

rev_system = (
    "You are a senior MongoDB Technical Services reviewer. "
    "Your job: validate an investigator's findings and produce a clean, safe final report.\n\n"
    "RULES:\n"
    "- Cross-reference claims against the release notes context provided.\n"
    "- Reject unsafe advice: never recommend 'just increase the memory limit' or 'disable the guardrail'.\n"
    "- Distinguish fact (from evidence) vs inference vs recommendation.\n"
    "- Note what is Atlas-only vs Enterprise vs Community scoped.\n"
    "- Flag when Engineering escalation is needed.\n"
    "- Keep the report concise.\n\n"
    "OUTPUT FORMAT:\n"
    "# Support Findings: MongoDB 9.0 Upgrade\n"
    "## Customer Symptom\n"
    "## Likely 9.0 Change\n"
    "## Evidence Collected\n"
    "## Scope (Atlas / Enterprise / Community)\n"
    "## Risk Assessment (Low / Medium / High)\n"
    "## Immediate Mitigation\n"
    "## Post-Upgrade Validation\n"
    "## Escalation Criteria\n"
    "## Sources (cite docs sections and error codes)"
)

# Cross-reference: search docs to validate investigator claims
validation_docs = search_docs("per-operation memory limit compatibility dotted path null BSON validation")
# The reviewer independently searches docs to verify claims.

rev_prompt = (
    f"INVESTIGATOR FINDINGS:\n{raw_findings}\n\n"
    f"RELEASE NOTES (FOR CROSS-REFERENCE):\n{validation_docs}\n\n"
    "Produce the structured support report now. If any finding cannot be verified from "
    "the release notes, say 'Unverified — needs further investigation.'"
)

rev_response = ollama.chat(model=MODEL, messages=[
    {"role": "system", "content": rev_system},
    {"role": "user", "content": rev_prompt},
])
# Second LLM call; the reviewer produces the final report.

final_report = rev_response["message"]["content"].strip()

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(final_report)
# Save the final reviewed report.

print("=== FINAL REPORT (saved to support_findings.md) ===\n")
print(final_report)
# Structured report: symptom, evidence, risk, mitigation, escalation. Ready for a customer.

=== TS REVIEWER AGENT STARTED ===

=== FINAL REPORT (saved to support_findings.md) ===

# Support Findings: MongoDB 9.0 Upgrade

## Customer Symptom
The symptoms observed indicate that queries are being throttled due to hitting the per-operation memory guardrail in MongoDB 9.0. The error codes `146` (ExceededMemoryLimit) and `292` (QueryExceededMemoryLimitNoDiskUseAllowed) indicate that operations are failing due to exceeding the available memory limit, which is 1 gigabyte for a single query operation.

## Likely 9.0 Changes
Starting from MongoDB 9.0, the server introduces a limit on the amount of memory that a single query operation can use, with the default limit being 1 gigabyte or 20% of the server's memory, whichever is greater. If operations exceed this limit, they fail with error codes `146` or `292`.

## Evidence Collected
- **Diagnostic Evidence**: The `explain()` operation shows that queries are hitting the memory guardrail, specifically, the peak memory usage is 113 MB. This

## Read the saved report

In [5]:
with open(OUTPUT_FILE, encoding="utf-8") as f:
    print(f.read())

# Support Findings: MongoDB 9.0 Upgrade

## Customer Symptom
The symptoms observed indicate that queries are being throttled due to hitting the per-operation memory guardrail in MongoDB 9.0. The error codes `146` (ExceededMemoryLimit) and `292` (QueryExceededMemoryLimitNoDiskUseAllowed) indicate that operations are failing due to exceeding the available memory limit, which is 1 gigabyte for a single query operation.

## Likely 9.0 Changes
Starting from MongoDB 9.0, the server introduces a limit on the amount of memory that a single query operation can use, with the default limit being 1 gigabyte or 20% of the server's memory, whichever is greater. If operations exceed this limit, they fail with error codes `146` or `292`.

## Evidence Collected
- **Diagnostic Evidence**: The `explain()` operation shows that queries are hitting the memory guardrail, specifically, the peak memory usage is 113 MB. This suggests that even though the operation is within the guardrail, the query is using a s

## Recap

- **Investigator** gathered evidence: searched docs, ran diagnostics, classified errors
- **TS Reviewer** validated findings: cross-referenced release notes, rejected unsafe advice, produced structured report
- The handoff is just passing text from one agent to the other — **our code** controls the workflow
- Two specialized roles beat one general agent for complex investigations

### What we built across all 3 notebooks
1. **RAG** — retrieve relevant docs, generate answer (no decisions)
2. **Agent** — model chooses tools, investigates autonomously
3. **Multi-agent** — role separation with handoff and validation